In [96]:
import pandas as pd

df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Types of data

In [97]:
# We see the types and see if there are any null or duplicated values
print(df.dtypes)

nulls = df.isnull().sum()
print(nulls[nulls > 0])

duplicates = df.duplicated().sum()
if duplicates > 0:
    print(f"Duplicated rows: {duplicates}")
else:
    print("No duplicated rows")

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object
Series([], dtype: int64)
No duplicated rows


In [98]:
df = df.dropna(subset=["TotalCharges"])  # Clean the null data

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)  # Str type to numeric value
df["TotalCharges"].dtype

# Now they are real NaN, so we can handle them (new customers, tenure=0, no charges yet)
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [99]:
# We want to see if the columns that have Yes/No have more values to clean them
for col in df.columns:
    values = df[col].dropna().unique()  # We store the values of each column

    if "Yes" in values or "No" in values:  # It checks the columns that have Yes/No
        print(f"\nColumn: {col}")  # We print the values
        print(values)


Column: Partner
<StringArray>
['Yes', 'No']
Length: 2, dtype: str

Column: Dependents
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

Column: PhoneService
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

Column: MultipleLines
<StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str

Column: InternetService
<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str

Column: OnlineSecurity
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

Column: OnlineBackup
<StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str

Column: DeviceProtection
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

Column: TechSupport
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

Column: StreamingTV
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

Column: StreamingMovies
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

Column: PaperlessBilling
<Stri

In [100]:
df["MultipleLines"].unique()

<StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str

In [101]:
df = df.replace({"No phone service": "No"})
df["MultipleLines"].value_counts()

MultipleLines
No     4072
Yes    2971
Name: count, dtype: int64

In [102]:
# We want to see if the columns that have No internet service have more values to clean them
no_internet_services = [
    col for col in df.columns if "No internet service" in df[col].values
]
print(no_internet_services)

['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']


In [103]:
df = df.replace({"No internet service": "No"})

no_internet_services = [
    col for col in df.columns if "No internet service" in df[col].values
]
print(no_internet_services)

[]


### Data Cleaning

* Checked missing values and duplicates
* Converted TotalCharges to numeric format
* Validated data type categories
* Transformed all data to Yes / No for a later conversion to binary

#### Yes/No to binary

In [104]:
# We need to convert Yes/No into binary for ML design
yes_no_columns = []

for col in df.columns:
    values = df[col].dropna()  # Store the no null values

    if values.isin(["Yes", "No"]).all():
        yes_no_columns.append(col)

for col in yes_no_columns:
    df[col] = df[col].map({"Yes": 1, "No": 0})  # Conversion to binary

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,1,0,1,0,0,DSL,0,...,0,0,0,0,Month-to-month,1,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,0,0,34,1,0,DSL,1,...,1,0,0,0,One year,0,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,0,0,2,1,0,DSL,1,...,0,0,0,0,Month-to-month,1,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,0,0,45,0,0,DSL,1,...,1,1,0,0,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,0,0,2,1,0,Fiber optic,0,...,0,0,0,0,Month-to-month,1,Electronic check,70.70,151.65,1


ML preparation

In [105]:
# Not a predictive feature - unique per row, would create noise in the model
df = df.drop(columns=["customerID"])

In [106]:
df = pd.get_dummies(df, drop_first=True)
df.head()

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,...,TotalCharges,Churn,gender_Male,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,0,1,0,0,0,1,0,0,...,29.85,0,False,False,False,False,False,False,True,False
1,0,0,0,34,1,0,1,0,1,0,...,1889.50,0,True,False,False,True,False,False,False,True
2,0,0,0,2,1,0,1,1,0,0,...,108.15,1,True,False,False,False,False,False,False,True
3,0,0,0,45,0,0,1,0,1,1,...,1840.75,0,True,False,False,True,False,False,False,False
4,0,0,0,2,1,0,0,0,0,0,...,151.65,1,False,True,False,False,False,False,True,False


## Checks

In [107]:
# Structural
assert df.isnull().sum().sum() == 0, "There are still null values in the dataset"
assert "costumerID" not in df.columns  # CostumerID removed
assert df.select_dtypes(include="object").empty  # Text removed

# Business logic
assert (df["MonthlyCharges"] >= 0).all()  # Negative values
assert (df["TotalCharges"] >= 0).all()

In [108]:
# We save the clean and prepare data
df.to_csv("../data/processed/clean_data.csv", index=False)